# TuranAssist – multilingual-e5-small, прогон v2

Второй прогон по итогам первого (`turanassist_e5_intents.ipynb`):

1. **Эмбеддинги всех наборов** (замороженный e5) выгружаются на Hugging Face – калибровка порога
   и ансамбль с TF-IDF считаются дальше локально в Docker, без GPU.
2. **Корректное дообучение:** голова на усреднении токенов (mean pooling, как обучалась e5),
   а не на CLS; отдельные скорости обучения для энкодера и головы, больше эпох.
3. **Варианты квантизации** экспортированного энкодера: полная int8, int8 с поканальными весами,
   int8 только для матрицы эмбеддингов. Для каждого: размер, память сессии (в отдельном процессе),
   задержка на CPU в один поток и расхождение с fp32.
4. Всё загружается в репозиторий модели в папку `v2/` – файлы первого прогона не трогаются.

**Перед запуском:** GPU T4, секрет `HF_TOKEN` с доступом из блокнота (как в первом прогоне).
Полный прогон – около 20–25 минут.

In [ ]:
# настройки запуска
GITHUB_REPO = "https://github.com/AlexTrav/TuranAssist.git"
HF_REPO = "AlexCode2003/turanassist-intent-e5"
BASE_MODEL = "intfloat/multilingual-e5-small"
SEED = 42
MAX_LEN = 64
BATCH_SIZE = 32
FT_EPOCHS = 15
FT_LR_ENCODER = 2e-5   # энкодер дообучаем бережно – он уже хорошо понимает язык
FT_LR_HEAD = 1e-3      # голова обучается с нуля – ей нужна скорость побольше
EXPORT_DIR = "/content/export_v2"

In [ ]:
!pip -q install transformers onnx onnxruntime huggingface_hub scikit-learn pyyaml psutil
!rm -rf /content/TuranAssist && git clone -q --depth 1 {GITHUB_REPO} /content/TuranAssist

import os, sys
os.environ["DATA_DIR"] = "/content/TuranAssist/data"
sys.path.insert(0, "/content/TuranAssist")

In [ ]:
import copy, json, random, subprocess, time
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from model.common import data, metrics

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("устройство:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

ood_val, ood_test = data.load_ood()
sets = {"train": data.load_train(), "test": data.load_test(), "ood_val": ood_val, "ood_test": ood_test,
        "external": data.load_external(), "scenarios": data.load_scenarios()}
TEXTS = {k: [e.text for e in v] for k, v in sets.items()}
y = [e.labels[0] for e in sets["train"]]
INTENTS = sorted(set(y))
label2id = {l: i for i, l in enumerate(INTENTS)}
y_ids = np.array([label2id[l] for l in y])
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=data.SEED).split(TEXTS["train"], y))
os.makedirs(f"{EXPORT_DIR}/dumps", exist_ok=True)
json.dump(TEXTS, open(f"{EXPORT_DIR}/dumps/texts.json", "w", encoding="utf-8"), ensure_ascii=False)
print({k: len(v) for k, v in sets.items()}, "интентов:", len(INTENTS))

## Общие функции

In [ ]:
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def encode(texts, tensors="pt"):
    return tokenizer(["query: " + t for t in texts], padding=True, truncation=True, max_length=MAX_LEN,
                     return_tensors=tensors)

def mean_pool(hidden, mask):
    mask = mask.unsqueeze(-1).to(hidden.dtype)
    return (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

@torch.no_grad()
def embed(encoder, texts, bs=128):
    out = []
    for i in range(0, len(texts), bs):
        batch = {k: v.to(device) for k, v in encode(texts[i:i + bs]).items()}
        hidden = encoder(**batch).last_hidden_state
        out.append(F.normalize(mean_pool(hidden, batch["attention_mask"]), dim=-1).cpu())
    return torch.cat(out).numpy()

# оценка по готовым вероятностям для каждого набора – так же, как у baseline (model/common/metrics.py)
def evaluate_probs(P, oof_proba, threshold=None):
    oof_correct = [INTENTS[i] == t for i, t in zip(oof_proba.argmax(1), y)]
    result = {"cv_accuracy": float(np.mean(oof_correct))}
    if threshold is None:
        threshold, info = metrics.choose_threshold(oof_correct, oof_proba.max(1), P["ood_val"].max(1))
        result["threshold_selection"] = info
    result["threshold"] = float(threshold)
    for key in ("test", "ood_test", "external", "scenarios"):
        pred, conf = [INTENTS[i] for i in P[key].argmax(1)], P[key].max(1)
        result[key] = metrics.evaluate_set(sets[key], pred, conf, threshold)
        if key == "test":
            true = [e.labels[0] for e in sets[key]]
            f1, per_intent = metrics.macro_f1(true, pred, INTENTS)
            result[key].update({"macro_f1_no_threshold": f1, "per_intent": per_intent,
                                "confusions": metrics.top_confusions(true, pred)})
    return result

def summary(name, r):
    t = r["test"]
    print(f"{name}: cv={r['cv_accuracy']:.3f} test={t['in_domain_top1_no_threshold']['value']:.3f} "
          f"f1={t['macro_f1_no_threshold']:.3f} thr={r['threshold']:.2f} "
          f"ood={r['ood_test']['ood_rejected']['value']:.3f} ext={r['external']['overall']['value']:.3f} "
          f"scen={r['scenarios']['overall']['value']:.3f}")

results = {}

## Замороженный e5: эмбеддинги всех наборов + логистическая регрессия (для сравнения с v1)

In [ ]:
encoder = AutoModel.from_pretrained(BASE_MODEL).to(device).eval()
EMB = {k: embed(encoder, TEXTS[k]) for k in sets}
np.savez_compressed(f"{EXPORT_DIR}/dumps/encoder_torch_fp32.npz", **EMB)

best_c, best_acc, oof_frozen = None, -1.0, None
for c in (1, 10, 100, 1000):
    oof = np.zeros((len(y), len(INTENTS)))
    for tr, va in folds:
        oof[va] = LogisticRegression(C=c, max_iter=5000).fit(EMB["train"][tr], y_ids[tr]).predict_proba(EMB["train"][va])
    acc = float(np.mean(oof.argmax(1) == y_ids))
    print(f"C={c}: cv accuracy {acc:.3f}")
    if acc > best_acc:
        best_c, best_acc, oof_frozen = c, acc, oof
head = LogisticRegression(C=best_c, max_iter=5000).fit(EMB["train"], y_ids)
results["e5_frozen_logreg"] = evaluate_probs({k: head.predict_proba(EMB[k]) for k in sets}, oof_frozen)
results["e5_frozen_logreg"]["C"] = best_c
summary("e5_frozen_logreg", results["e5_frozen_logreg"])

## Дообучение с головой на усреднении токенов

В первом прогоне голова стояла на CLS-токене, а e5 обучена на усреднении – дообучение не сошлось.
Здесь голова получает усреднённый вектор (как при обучении e5), а энкодер и голова учатся с разными скоростями.

In [ ]:
class MeanPoolClassifier(torch.nn.Module):
    def __init__(self, n_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(BASE_MODEL)
        self.dropout = torch.nn.Dropout(0.1)
        self.head = torch.nn.Linear(self.encoder.config.hidden_size, n_labels)

    def forward(self, input_ids, attention_mask):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        return self.head(self.dropout(mean_pool(hidden, attention_mask)))

def finetune(texts, labels, seed=SEED):
    torch.manual_seed(seed)
    model = MeanPoolClassifier(len(INTENTS)).to(device)
    enc = encode(list(texts))
    dataset = torch.utils.data.TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(labels))
    loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                                         generator=torch.Generator().manual_seed(seed))
    optimizer = torch.optim.AdamW([{"params": model.encoder.parameters(), "lr": FT_LR_ENCODER},
                                   {"params": model.head.parameters(), "lr": FT_LR_HEAD}], weight_decay=0.01)
    total = FT_EPOCHS * len(loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total), total)
    model.train()
    for epoch in range(FT_EPOCHS):
        for ids, mask, lab in loader:
            logits = model(ids.to(device), mask.to(device))
            loss = F.cross_entropy(logits, lab.to(device))
            loss.backward()
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
    return model.eval()

@torch.no_grad()
def finetuned_proba(model, texts, bs=128):
    out = []
    for i in range(0, len(texts), bs):
        batch = encode(texts[i:i + bs])
        out.append(torch.softmax(model(batch["input_ids"].to(device), batch["attention_mask"].to(device)), -1).cpu())
    return torch.cat(out).numpy()

start = time.time()
X_arr = np.array(TEXTS["train"], dtype=object)
oof_ft = np.zeros((len(y), len(INTENTS)))
for k, (tr, va) in enumerate(folds, 1):
    fold_model = finetune(X_arr[tr], y_ids[tr])
    oof_ft[va] = finetuned_proba(fold_model, list(X_arr[va]))
    print(f"фолд {k}: accuracy {np.mean(oof_ft[va].argmax(1) == y_ids[va]):.3f}")
    del fold_model; torch.cuda.empty_cache()

ft_model = finetune(X_arr, y_ids)
FT = {k: finetuned_proba(ft_model, TEXTS[k]) for k in sets}
np.savez_compressed(f"{EXPORT_DIR}/dumps/finetuned_torch_fp32.npz", train_oof=oof_ft, **FT)
results["e5_finetuned_meanpool"] = evaluate_probs(FT, oof_ft)
results["e5_finetuned_meanpool"]["train_minutes_cv_and_final"] = (time.time() - start) / 60
summary("e5_finetuned_meanpool", results["e5_finetuned_meanpool"])

## Экспорт в ONNX и варианты квантизации

Энкодер (усреднённый нормированный эмбеддинг) экспортируется всегда – он нужен для ансамбля с TF-IDF.
Дообученная модель (вероятности интентов) – если по кросс-валидации она лучше замороженной.

In [ ]:
class EncoderEmbedding(torch.nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def forward(self, input_ids, attention_mask):
        hidden = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        return F.normalize(mean_pool(hidden, attention_mask), dim=-1)

class FinetunedProba(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask):
        return torch.softmax(self.model(input_ids, attention_mask), dim=-1)

def export_onnx(module, path, output_name):
    sample = encode(["пример вопроса"])
    inputs = (sample["input_ids"], sample["attention_mask"])
    args = dict(input_names=["input_ids", "attention_mask"], output_names=[output_name], opset_version=17,
                dynamic_axes={"input_ids": {0: "batch", 1: "seq"}, "attention_mask": {0: "batch", 1: "seq"},
                              output_name: {0: "batch"}})
    module = module.cpu().eval()
    try:  # классический экспортёр TorchScript – самый предсказуемый для BERT-подобных моделей
        torch.onnx.export(module, inputs, path, dynamo=False, **args)
    except TypeError:
        torch.onnx.export(module, inputs, path, **args)
    except Exception as exc:
        print("TorchScript-экспорт недоступен:", exc)
        torch.onnx.export(module, inputs, path, dynamo=True, **args)

graphs = {"encoder": (EncoderEmbedding(copy.deepcopy(encoder)), "embedding")}
if results["e5_finetuned_meanpool"]["cv_accuracy"] > results["e5_frozen_logreg"]["cv_accuracy"]:
    graphs["finetuned"] = (FinetunedProba(copy.deepcopy(ft_model)), "probabilities")
print("экспортируем:", list(graphs))

from onnxruntime.quantization import QuantType, quantize_dynamic
QUANT = {
    "int8_full": dict(weight_type=QuantType.QInt8),
    "int8_full_perchannel": dict(weight_type=QuantType.QInt8, per_channel=True),
    "int8_emb_only": dict(weight_type=QuantType.QInt8, op_types_to_quantize=["Gather"]),
}
FILES = {}
for g, (module, out_name) in graphs.items():
    fp32 = f"{EXPORT_DIR}/{g}_fp32.onnx"
    export_onnx(module, fp32, out_name)
    FILES[(g, "fp32")] = fp32
    for v, kwargs in QUANT.items():
        path = f"{EXPORT_DIR}/{g}_{v}.onnx"
        quantize_dynamic(fp32, path, **kwargs)
        FILES[(g, v)] = path
tokenizer.save_pretrained(EXPORT_DIR)
for (g, v), p in FILES.items():
    print(f"{g}/{v}: {os.path.getsize(p) / 2**20:.0f} МБ")

## Сравнение вариантов: размер, память, задержка, расхождение с fp32

Задержка – одиночный запрос (токенизация + модель) на CPU в один поток. Память сессии меряется
в отдельном процессе, чтобы на замер не влияли уже загруженные модели.

In [ ]:
import onnxruntime as ort

RSS_SCRIPT = "/content/rss_probe.py"
open(RSS_SCRIPT, "w").write(
    "import sys, psutil, numpy as np, onnxruntime as ort\n"
    "p = psutil.Process(); before = p.memory_info().rss\n"
    "o = ort.SessionOptions(); o.intra_op_num_threads = 1; o.inter_op_num_threads = 1\n"
    "s = ort.InferenceSession(sys.argv[1], o, providers=['CPUExecutionProvider'])\n"
    "ids = np.ones((1, 16), dtype=np.int64)\n"
    "s.run(None, {'input_ids': ids, 'attention_mask': ids})\n"
    "print((p.memory_info().rss - before) / 2**20)\n")

def session(path):
    o = ort.SessionOptions(); o.intra_op_num_threads = 1; o.inter_op_num_threads = 1
    return ort.InferenceSession(path, o, providers=["CPUExecutionProvider"])

def run(sess, texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        enc = encode(texts[i:i + bs], tensors="np")
        out.append(sess.run(None, {"input_ids": enc["input_ids"].astype(np.int64),
                                   "attention_mask": enc["attention_mask"].astype(np.int64)})[0])
    return np.concatenate(out)

quant_report = {}
for (g, v), path in FILES.items():
    sess = session(path)
    OUT = {k: run(sess, TEXTS[k]) for k in sets}
    np.savez_compressed(f"{EXPORT_DIR}/dumps/{g}_{v}.npz", **OUT)
    rss = float(subprocess.run([sys.executable, RSS_SCRIPT, path], capture_output=True, text=True).stdout.strip() or "nan")
    lat = metrics.latency_ms(lambda t: run(sess, [t]), TEXTS["test"])
    if g == "encoder":
        ref = EMB
        cos = np.concatenate([(OUT[k] * ref[k]).sum(1) for k in sets])
        probs = {k: head.predict_proba(OUT[k]) for k in sets}
        fidelity = {"cos_mean": float(cos.mean()), "cos_min": float(cos.min())}
        ref_probs = {k: head.predict_proba(ref[k]) for k in sets}
        oof, thr = oof_frozen, results["e5_frozen_logreg"]["threshold"]
    else:
        probs, ref_probs = OUT, FT
        fidelity = {"max_abs_diff": float(max(np.abs(OUT[k] - FT[k]).max() for k in sets))}
        oof, thr = oof_ft, results["e5_finetuned_meanpool"]["threshold"]
    agree = float(np.mean(np.concatenate([probs[k].argmax(1) == ref_probs[k].argmax(1) for k in sets])))
    ev = evaluate_probs(probs, oof, threshold=thr)
    quant_report[f"{g}/{v}"] = {"size_mb": os.path.getsize(path) / 2**20, "session_rss_mb": rss,
                                "latency_ms": lat, "top1_agreement_vs_fp32": agree, **fidelity, "eval": ev}
    print(f"{g}/{v}: {os.path.getsize(path) / 2**20:.0f} МБ, RSS {rss:.0f} МБ, p50 {lat['p50']:.1f} мс, "
          f"p95 {lat['p95']:.1f} мс, совпадение top-1 {agree:.3f}, {fidelity}, "
          f"test {ev['test']['in_domain_top1_no_threshold']['value']:.3f}, scen {ev['scenarios']['overall']['value']:.3f}")

## Отчёт v2 и загрузка на Hugging Face

Файлы fp32 не загружаются (450 МБ и больше, бэкенду не нужны). Токен – из Colab Secrets, не выводится.

In [ ]:
def fmt(p):
    return f"{p['value']:.3f} [{p['ci95'][0]:.3f}; {p['ci95'][1]:.3f}]"

lines = ["# multilingual-e5-small, прогон v2", "",
         "Значения – доля верных ответов, в скобках 95% доверительный интервал Уилсона.", "",
         "## Замороженный e5 против корректного дообучения (PyTorch fp32)", "",
         "| Модель | CV acc | Test top-1 | macro-F1 | Порог | Test с порогом | OOD отклонено | Внешний тест | Сценарии |",
         "|---|---|---|---|---|---|---|---|---|"]
for name, r in results.items():
    lines.append(f"| `{name}` | {r['cv_accuracy']:.3f} | {fmt(r['test']['in_domain_top1_no_threshold'])} "
                 f"| {r['test']['macro_f1_no_threshold']:.3f} | {r['threshold']:.3f} | {fmt(r['test']['overall'])} "
                 f"| {fmt(r['ood_test']['ood_rejected'])} | {fmt(r['external']['overall'])} | {fmt(r['scenarios']['overall'])} |")
lines += ["", "## Варианты ONNX: размер, память, задержка (CPU, 1 поток), качество", "",
          "| Вариант | Размер, МБ | Память сессии, МБ | p50, мс | p95, мс | Совпадение top-1 с fp32 | Test top-1 | Сценарии с порогом |",
          "|---|---|---|---|---|---|---|---|"]
for name, q in quant_report.items():
    e = q["eval"]
    lines.append(f"| `{name}` | {q['size_mb']:.0f} | {q['session_rss_mb']:.0f} | {q['latency_ms']['p50']:.1f} "
                 f"| {q['latency_ms']['p95']:.1f} | {q['top1_agreement_vs_fp32']:.3f} "
                 f"| {fmt(e['test']['in_domain_top1_no_threshold'])} | {fmt(e['scenarios']['overall'])} |")
report_md = "\n".join(lines) + "\n"
print(report_md)
open(f"{EXPORT_DIR}/report_v2.md", "w", encoding="utf-8").write(report_md)
json.dump({"results": results, "quantization": quant_report, "intents": INTENTS, "frozen_C": best_c,
           "settings": {"max_length": MAX_LEN, "query_prefix": "query: ", "ft_epochs": FT_EPOCHS,
                        "ft_lr_encoder": FT_LR_ENCODER, "ft_lr_head": FT_LR_HEAD, "seed": SEED}},
          open(f"{EXPORT_DIR}/metrics_v2.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2, default=float)

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

api = HfApi(token=userdata.get("HF_TOKEN"))
api.upload_folder(folder_path=EXPORT_DIR, path_in_repo="v2", repo_id=HF_REPO, repo_type="model",
                  ignore_patterns=["*_fp32.onnx"], commit_message="Прогон v2: эмбеддинги, дообучение, варианты квантизации")
print("готово: https://huggingface.co/" + HF_REPO + "/tree/main/v2")